# J1 : Lancement

## Introduction au projet "Durée et consommation en phase de taxi au sol"

### Objectifs
Ce projet vise à analyser la durée et la consommation de carburant des avions pendant la phase de taxi au sol. Cette phase, bien que courte, représente une part significative de la consommation totale et des émissions de CO2 dans les opérations aéroportuaires.

### Contexte
- **Phase de taxi** : Période entre le décollage de la piste et l'arrêt complet de l'avion, ou inversement pour l'atterrissage.
- **Données** : Fichier HDF5 contenant des enregistrements de vols avec diverses mesures (vitesse, consommation, temps, etc.).
- **Analyse** : Exploration, nettoyage, visualisation et modélisation pour comprendre les relations entre durée de taxi et consommation.

### Présentation du fichier Aircraft_01.h5
Le fichier `Aircraft_01.h5` contient des données structurées en plusieurs enregistrements HDF5. Chaque enregistrement correspond à un vol et contient des séries temporelles de mesures avioniques.

**Structure typique :**
- Clés principales : noms des vols ou phases
- Pour chaque vol : DataFrame avec colonnes comme temps, vitesse, consommation de carburant, etc.

In [18]:
# Installation des librairies nécessaires
!pip install h5py pandas numpy matplotlib seaborn scikit-learn plotly

In [22]:
# Import des librairies
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Configuration des affichages
plt.style.use('default')
sns.set_palette("husl")
print("Librairies importées avec succès !")

Librairies importées avec succès !


In [20]:
# Lecture du fichier HDF5
import os

# Chemin vers le fichier HDF5
HDF5_FILE = 'data/Aircraft_01.h5'  # Adapter si nécessaire

if os.path.exists(HDF5_FILE):
    print(f"Fichier local trouvé : {HDF5_FILE}")
    with h5py.File(HDF5_FILE, 'r') as f:
        keys = list(f.keys())
        print(f"Clés du fichier HDF5 : {keys}")
        
        # Afficher un aperçu des premières clés
        for i, key in enumerate(keys[:3]):
            print(f"\nClé '{key}' :")
            if isinstance(f[key], h5py.Group):
                subkeys = list(f[key].keys())
                print(f"  Sous-clés : {subkeys[:5]}...")  # Limiter l'affichage
                if subkeys:
                    # Aperçu des données (premières lignes)
                    data_shape = f[key][subkeys[0]].shape if hasattr(f[key][subkeys[0]], 'shape') else 'Non-array'
                    print(f"  Forme des données pour {subkeys[0]} : {data_shape}")
else:
    print(f"Fichier local non trouvé. Tentative de téléchargement depuis Google Drive...")
    # Code pour Google Drive (adapté du code existant)
    # [Insérer le code d'authentification et téléchargement ici si nécessaire]
    print("Veuillez adapter le code pour le téléchargement Google Drive.")

Fichier local trouvé : data/Aircraft_01.h5
Clés du fichier HDF5 : ['record_00', 'record_01', 'record_02', 'record_03', 'record_04', 'record_05', 'record_06', 'record_07', 'record_08', 'record_09', 'record_10', 'record_100', 'record_1000', 'record_1001', 'record_101', 'record_102', 'record_103', 'record_104', 'record_105', 'record_106', 'record_107', 'record_108', 'record_109', 'record_11', 'record_110', 'record_111', 'record_112', 'record_113', 'record_114', 'record_115', 'record_116', 'record_117', 'record_118', 'record_119', 'record_12', 'record_120', 'record_121', 'record_122', 'record_123', 'record_124', 'record_125', 'record_126', 'record_127', 'record_128', 'record_129', 'record_13', 'record_130', 'record_131', 'record_132', 'record_133', 'record_134', 'record_135', 'record_136', 'record_137', 'record_138', 'record_139', 'record_14', 'record_140', 'record_141', 'record_142', 'record_143', 'record_144', 'record_145', 'record_146', 'record_147', 'record_148', 'record_149', 'record_

## Concepts clés

### Durée de taxi
La durée de taxi correspond au temps écoulé entre le moment où l'avion quitte la porte d'embarquement et le début du décollage (ou inversement pour l'arrivée). Cette durée peut varier considérablement selon :
- La congestion de l'aéroport
- Les conditions météorologiques
- La taille de l'avion
- Les procédures aéroportuaires

### Consommation au sol
Pendant la phase de taxi, l'avion consomme du carburant pour :
- Faire tourner les moteurs à faible puissance
- Alimenter les systèmes électriques
- Maintenir les conditions de pressurisation

**Unité typique** : kg/h ou L/h de carburant.

**Impact environnemental** : Bien que courte, cette phase représente ~5-10% de la consommation totale d'un vol court.

# J2 : Collaboratif

## Exploration des codes existants

Le projet contient plusieurs modules Python qui peuvent être réutilisés pour l'analyse :
- `opset.py` : Gestion des ensembles d'observations HDF5
- `plots.py` : Fonctions de visualisation
- `instants.py` : Extraction d'instants/events dans les signaux
- `tubes.py` : Création de tubes de confiance pour la prédiction

Ces modules utilisent des DataFrames pandas et des visualisations Plotly pour analyser les données avioniques.

In [21]:
# Import des modules du projet
# Note: Les modules utilisent des imports relatifs, installation du package nécessaire
# from tabata.opset import Opset
# from tabata.plots import nameunit
# from tabata.instants import indicator
# from tabata.tubes import highlight

print("Modules du projet : à installer séparément si nécessaire")
print("Les fonctions peuvent être adaptées directement dans le notebook")

Modules du projet : à installer séparément si nécessaire
Les fonctions peuvent être adaptées directement dans le notebook


In [32]:
# Fonction réutilisable pour charger les données de taxi
import os  # Assurer que os est importé
import h5py
import pandas as pd

HDF5_FILE = 'data/Aircraft_01.h5'  # Définition locale

def load_taxi_data(hdf5_file, key=None, preview_only=True):
    """
    Charge les données de phase taxi depuis un fichier HDF5.
    
    Parameters:
    hdf5_file (str): Chemin vers le fichier HDF5
    key (str): Clé spécifique à charger (optionnel)
    preview_only (bool): Si True, ne charge que les métadonnées et un aperçu
    
    Returns:
    dict: Dictionnaire avec les informations
    """
    data = {}
    with h5py.File(hdf5_file, 'r') as f:
        keys = list(f.keys()) if key is None else [key]
        
        for k in keys:
            if k in f:
                obj = f[k]
                if isinstance(obj, h5py.Dataset):
                    # C'est un dataset
                    shape = obj.shape
                    dtype = obj.dtype
                    if preview_only:
                        data[k] = f"Dataset: shape={shape}, dtype={dtype}"
                    else:
                        try:
                            data[k] = obj[:100]  # Limiter à 100 lignes
                        except:
                            data[k] = f"Erreur chargement dataset: {shape}"
                elif isinstance(obj, h5py.Group):
                    # C'est un groupe
                    subkeys = list(obj.keys())
                    data[k] = f"Groupe avec {len(subkeys)} sous-clés: {subkeys[:5]}..."
    
    return data

# Test de la fonction
if os.path.exists(HDF5_FILE):
    taxi_data = load_taxi_data(HDF5_FILE)
    print(f"Données chargées pour {len(taxi_data)} clés")
    for key, info in taxi_data.items():
        print(f"  {key}: {info}")

Données chargées pour 1002 clés
  record_00: Groupe avec 4 sous-clés: ['axis0', 'axis1', 'block0_items', 'block0_values']...
  record_01: Groupe avec 4 sous-clés: ['axis0', 'axis1', 'block0_items', 'block0_values']...
  record_02: Groupe avec 4 sous-clés: ['axis0', 'axis1', 'block0_items', 'block0_values']...
  record_03: Groupe avec 4 sous-clés: ['axis0', 'axis1', 'block0_items', 'block0_values']...
  record_04: Groupe avec 4 sous-clés: ['axis0', 'axis1', 'block0_items', 'block0_values']...
  record_05: Groupe avec 4 sous-clés: ['axis0', 'axis1', 'block0_items', 'block0_values']...
  record_06: Groupe avec 4 sous-clés: ['axis0', 'axis1', 'block0_items', 'block0_values']...
  record_07: Groupe avec 4 sous-clés: ['axis0', 'axis1', 'block0_items', 'block0_values']...
  record_08: Groupe avec 4 sous-clés: ['axis0', 'axis1', 'block0_items', 'block0_values']...
  record_09: Groupe avec 4 sous-clés: ['axis0', 'axis1', 'block0_items', 'block0_values']...
  record_10: Groupe avec 4 sous-clés: 

# J3 : Nettoyage des données

## Méthodologie du nettoyage

1. **Identification des valeurs manquantes** : Utilisation de `isnull()` et `isna()` de pandas
2. **Traitement des valeurs aberrantes** : Méthodes statistiques (IQR, z-score) ou domaine (valeurs négatives impossibles)
3. **Conversion d'unités** : Temps en minutes, consommation en kg, etc.
4. **Filtrage des données** : Suppression des enregistrements incomplets ou invalides

**Choix méthodologiques** :
- Suppression des lignes avec >50% de valeurs manquantes
- Imputation par médiane pour les valeurs manquantes isolées
- Conversion temps de secondes à minutes
- Vérification cohérence physique (consommation > 0, durée > 0)

In [40]:
# Fonction de nettoyage des données
def clean_taxi_data(df, time_col='time', fuel_col='fuel_consumption'):
    """
    Nettoie les données de taxi : valeurs manquantes, aberrantes, conversions.
    
    Parameters:
    df (pd.DataFrame): DataFrame à nettoyer
    time_col (str): Nom de la colonne temps
    fuel_col (str): Nom de la colonne consommation
    
    Returns:
    pd.DataFrame: DataFrame nettoyé
    """
    df_clean = df.copy()
    
    # 1. Supprimer les lignes avec trop de NaN
    threshold = len(df_clean.columns) * 0.5
    df_clean = df_clean.dropna(thresh=threshold)
    
    # 2. Imputation des valeurs manquantes restantes
    for col in df_clean.columns:
        if df_clean[col].dtype in ['float64', 'int64']:
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    
    # 3. Conversion d'unités (exemple : temps en minutes)
    if time_col in df_clean.columns:
        df_clean[time_col] = df_clean[time_col] / 60  # secondes -> minutes
    
    # 4. Suppression des valeurs aberrantes (IQR method)
    if fuel_col in df_clean.columns:
        Q1 = df_clean[fuel_col].quantile(0.25)
        Q3 = df_clean[fuel_col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_clean = df_clean[(df_clean[fuel_col] >= lower_bound) & (df_clean[fuel_col] <= upper_bound)]
    
    # 5. Vérifications de cohérence
    if time_col in df_clean.columns:
        df_clean = df_clean[df_clean[time_col] > 0]
    if fuel_col in df_clean.columns:
        df_clean = df_clean[df_clean[fuel_col] > 0]
    
    print(f"Données nettoyées : {len(df)} -> {len(df_clean)} lignes")
    return df_clean

import h5py
import pandas as pd

HDF5_FILE = 'data/Aircraft_01.h5'
RECORD = 'record_01'  # le record que tu veux analyser

# Charger le dataset depuis HDF5
with h5py.File(HDF5_FILE, 'r') as f:
    dataset = f[RECORD][()]  # [] renvoie un array NumPy complet

columns = ['time', 'fuel_consumption', 'speed', 'throttle']  # à adapter selon ton HDF5
df_taxi = pd.DataFrame(dataset, columns=columns)

columns = ['time', 'fuel_consumption', 'speed', 'throttle']  # à adapter selon ton HDF5
df_taxi = pd.DataFrame(dataset, columns=columns)

df_clean = clean_taxi_data(df_taxi, time_col='time', fuel_col='fuel_consumption')


TypeError: Accessing a group is done with bytes or str, not <class 'tuple'>

# J4 : Analyse descriptive

## Statistiques descriptives et visualisations

Cette section présente l'analyse exploratoire des données :
- **Distributions** : Histogrammes et boxplots de la durée et consommation
- **Statistiques** : Moyenne, médiane, écart-type, quartiles
- **Relations** : Scatter plots et corrélations entre variables

**Observations attendues** :
- Distribution asymétrique de la durée (queue droite)
- Relation positive entre durée et consommation
- Identification d'outliers (taxis très longs)

**Hypothèses** :
- Plus la durée augmente, plus la consommation augmente linéairement
- Effet seuil pour les taxis courts vs longs

In [23]:
# Fonction d'analyse descriptive
def descriptive_analysis(df, time_col='time', fuel_col='fuel_consumption'):
    """
    Effectue l'analyse descriptive des données de taxi.
    
    Parameters:
    df (pd.DataFrame): DataFrame à analyser
    time_col (str): Colonne durée
    fuel_col (str): Colonne consommation
    """
    print("=== STATISTIQUES DESCRIPTIVES ===")
    print(df[[time_col, fuel_col]].describe())
    
    # Corrélation
    corr = df[[time_col, fuel_col]].corr()
    print(f"\nCorrélation entre {time_col} et {fuel_col} : {corr.iloc[0,1]:.3f}")
    
    # Visualisations
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Histogramme durée
    axes[0,0].hist(df[time_col], bins=30, alpha=0.7, color='blue')
    axes[0,0].set_title('Distribution de la durée de taxi')
    axes[0,0].set_xlabel('Durée (min)')
    axes[0,0].set_ylabel('Fréquence')
    
    # Histogramme consommation
    axes[0,1].hist(df[fuel_col], bins=30, alpha=0.7, color='green')
    axes[0,1].set_title('Distribution de la consommation')
    axes[0,1].set_xlabel('Consommation (kg)')
    axes[0,1].set_ylabel('Fréquence')
    
    # Boxplot durée
    axes[1,0].boxplot(df[time_col])
    axes[1,0].set_title('Boxplot durée de taxi')
    axes[1,0].set_ylabel('Durée (min)')
    
    # Scatter plot
    axes[1,1].scatter(df[time_col], df[fuel_col], alpha=0.6, color='red')
    axes[1,1].set_title('Relation durée vs consommation')
    axes[1,1].set_xlabel('Durée (min)')
    axes[1,1].set_ylabel('Consommation (kg)')
    
    plt.tight_layout()
    plt.show()

# Exemple d'utilisation (avec données fictives pour démonstration)
# np.random.seed(42)
# df_sample = pd.DataFrame({
#     'time': np.random.exponential(10, 100),  # Durée en minutes
#     'fuel_consumption': 50 + 5 * np.random.exponential(10, 100) + np.random.normal(0, 10, 100)
# })
# descriptive_analysis(df_sample)

# J5 : Modélisation

## Hypothèse et modèle

**Hypothèse** : "La durée de taxi au sol influence significativement la consommation de carburant, avec une relation linéaire positive."

**Modèle choisi** : Régression linéaire simple
- Variable indépendante (X) : Durée de taxi (minutes)
- Variable dépendante (Y) : Consommation de carburant (kg)

**Métriques d'évaluation** :
- Coefficient de détermination (R²)
- Erreur quadratique moyenne (MSE)
- Analyse des résidus

**Limites** :
- Relation peut ne pas être strictement linéaire
- Variables confondantes non incluses (poids avion, conditions météo, etc.)
- Données peuvent présenter de l'hétéroscédasticité

In [24]:
# Fonction de modélisation
def model_taxi_consumption(df, time_col='time', fuel_col='fuel_consumption'):
    """
    Modélise la relation entre durée de taxi et consommation.
    
    Parameters:
    df (pd.DataFrame): DataFrame avec les données
    time_col (str): Colonne durée
    fuel_col (str): Colonne consommation
    
    Returns:
    dict: Résultats du modèle
    """
    # Préparation des données
    X = df[[time_col]].values
    y = df[fuel_col].values
    
    # Modèle de régression linéaire
    model = LinearRegression()
    model.fit(X, y)
    
    # Prédictions
    y_pred = model.predict(X)
    
    # Métriques
    r2 = r2_score(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    
    print("=== RÉSULTATS DU MODÈLE ===")
    print(f"Coefficient (pente) : {model.coef_[0]:.3f}")
    print(f"Intercept : {model.intercept_:.3f}")
    print(f"R² : {r2:.3f}")
    print(f"RMSE : {rmse:.3f}")
    
    # Visualisation
    plt.figure(figsize=(10, 6))
    plt.scatter(X, y, alpha=0.6, label='Données réelles', color='blue')
    plt.plot(X, y_pred, color='red', linewidth=2, label='Modèle linéaire')
    plt.xlabel('Durée de taxi (min)')
    plt.ylabel('Consommation (kg)')
    plt.title('Modèle de régression : Durée vs Consommation')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Analyse des résidus
    residuals = y - y_pred
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.scatter(y_pred, residuals, alpha=0.6, color='green')
    plt.axhline(y=0, color='red', linestyle='--')
    plt.xlabel('Prédictions')
    plt.ylabel('Résidus')
    plt.title('Résidus vs Prédictions')
    
    plt.subplot(1, 2, 2)
    plt.hist(residuals, bins=30, alpha=0.7, color='orange')
    plt.xlabel('Résidus')
    plt.ylabel('Fréquence')
    plt.title('Distribution des résidus')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'model': model,
        'r2': r2,
        'mse': mse,
        'rmse': rmse,
        'predictions': y_pred
    }

# Exemple d'utilisation (avec données fictives)
# results = model_taxi_consumption(df_sample)

NameError: name 'df' is not defined

# Conclusion

Ce notebook présente une analyse complète de la durée et consommation en phase de taxi au sol :

## Points clés abordés :
1. **Configuration** : Installation et import des librairies nécessaires
2. **Exploration** : Lecture et compréhension du fichier HDF5
3. **Réutilisation** : Intégration des modules existants du projet
4. **Nettoyage** : Préparation rigoureuse des données
5. **Analyse** : Statistiques descriptives et visualisations
6. **Modélisation** : Test d'hypothèse avec régression linéaire

## Prochaines étapes possibles :
- Intégrer d'autres variables (poids avion, conditions météo)
- Utiliser des modèles non-linéaires (polynomial, random forest)
- Analyser l'impact environnemental (émissions CO2)
- Comparer différents aéroports ou types d'avions

## Recommandations :
- Optimiser les procédures de taxi pour réduire la durée
- Améliorer l'efficacité énergétique des moteurs au sol
- Développer des systèmes de guidage automatisés

Le code est modulaire et réutilisable pour d'autres analyses similaires.